In [ ]:
%%capture

# Installing SpeechBrain via pip
BRANCH = 'develop'
!python -m pip install git+https://github.com/speechbrain/speechbrain.git@$BRANCH

In [9]:
from speechbrain.dataio.dataio import read_audio
signal = read_audio('/home/sayed/corpus/cmu_kids/kids/fabm/signal/fabm2al1.wav')
clean = signal.unsqueeze(0) # [batch, time]
from IPython.display import Audio
Audio('/home/sayed/corpus/cmu_kids/kids/fabm/signal/fabm2al1.wav')



In [10]:
with open("/home/sayed/espnet/egs/CHILD_ASR/LISTS/in_child_wav.list","r") as org_file:
    org_items = [line.strip() for line in org_file if line.strip()]

with open("/home/sayed/espnet/egs/CHILD_ASR/LISTS/out_child_wav_frequency_drop.list","r") as freqdrop_file:
    freqdrop_items = [line.strip() for line in freqdrop_file if line.strip()]
    
#print(len(org_items),org_items[0],type(org_items[0]),crop_items[0],type(crop_items[0]))

In [11]:
from speechbrain.augment.time_domain import DropFreq

dropper = DropFreq(drop_freq_count_low=5, drop_freq_count_high=8)
dropped_signal = dropper(clean)

Audio(dropped_signal,rate=16000)

In [12]:
import soundfile as sf
import numpy as np

In [19]:
for i in range(len(org_items)):
   signal = read_audio(org_items[i])
   clean = signal.unsqueeze(0) # [batch, time]

   dropper = DropFreq(drop_freq_count_low=5, drop_freq_count_high=8)
   dropped_signal = dropper(clean)

   augmented_data = np.array(dropped_signal,dtype=np.float32)

   if len(augmented_data.shape) >1:
      augmented_data_new = augmented_data.flatten()

   sf.write(freqdrop_items[i],augmented_data_new,16000)

In [14]:
import torch
from speechbrain.augment.time_domain import DoClip

clipper = DoClip(clip_low=0.7, clip_high=0.7)
#sinusoid = torch.sin(torch.linspace(0,20, 300))
clipped_signal = clipper(clean)

Audio(clipped_signal,rate=16000)

In [15]:
with open("/home/sayed/espnet/egs/CHILD_ASR/LISTS/out_child_wav_frequency_drop.list","r") as clip_file:
    clip_items = [line.strip() for line in clip_file if line.strip()]
    

In [16]:
for i in range(len(org_items)):
   testsignal = read_audio(org_items[i])
   clean = testsignal.unsqueeze(0) # [batch, time]

   clipper = DoClip(clip_low=0.7, clip_high=0.7)   
   clipped_signal = clipper(clean)

   augmented_data = np.array(clipped_signal,dtype=np.float32)

   if len(augmented_data.shape) >1:
      augmented_data_new = augmented_data.flatten()

   sf.write(clip_items[i],augmented_data_new,16000)